# Make y Zapier con IA: Automatización Low-Code

> Aprende a integrar Claude con plataformas de automatización como Make (ex-Integromat)
> y Zapier a través de HTTP modules y webhooks.

## Objetivos
- Entender cómo Make y Zapier conectan con APIs externas
- Simular escenarios Make con módulos HTTP + Claude
- Construir un Zap de procesamiento de contenido
- Implementar patrones de transformación de datos con LLM

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic requests pandas --quiet

## 2. Make vs Zapier vs n8n: comparativa

In [ ]:
import anthropic
import json
import pandas as pd
from datetime import datetime

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

comparativa = pd.DataFrame([
    {"Plataforma": "Make (Integromat)", "Plan Gratuito": "1000 ops/mes", "Precio Base": "9€/mes",
     "Fortaleza": "Flujos complejos, transformaciones", "Integración IA": "HTTP module → Anthropic API",
     "Open Source": "No"},
    {"Plataforma": "Zapier", "Plan Gratuito": "100 tareas/mes", "Precio Base": "20$/mes",
     "Fortaleza": "Más integraciones (6000+)", "Integración IA": "HTTP action / Webhooks",
     "Open Source": "No"},
    {"Plataforma": "n8n", "Plan Gratuito": "Self-hosted ilimitado", "Precio Base": "20€/mes (cloud)",
     "Fortaleza": "Control total, código personalizado", "Integración IA": "Nodo HTTP / código Python/JS",
     "Open Source": "Sí"},
    {"Plataforma": "Power Automate", "Plan Gratuito": "Con Microsoft 365", "Precio Base": "15$/user/mes",
     "Fortaleza": "Ecosistema Microsoft", "Integración IA": "Azure OpenAI / HTTP",
     "Open Source": "No"},
])

print("=== COMPARATIVA DE PLATAFORMAS DE AUTOMATIZACIÓN ===")
print(comparativa.to_string(index=False))
print("""
Cómo conectar Claude desde cualquiera de estas plataformas:

1. Crear módulo/acción HTTP
2. URL: https://api.anthropic.com/v1/messages
3. Método: POST
4. Headers:
   - x-api-key: {{ANTHROPIC_API_KEY}}
   - anthropic-version: 2023-06-01
   - content-type: application/json
5. Body JSON con model, max_tokens y messages
6. Mapear campo: choices[0].message.content → siguiente módulo
""")

## 3. Escenario Make: Pipeline de contenido RSS → Newsletter

In [ ]:
# Simular artículos RSS (en producción: módulo RSS de Make)
ARTICULOS_RSS = [
    {"titulo": "Meta lanza Llama 4 con contexto de 10M tokens",
     "fuente": "TechCrunch", "url": "https://techcrunch.com/llama4",
     "contenido": "Meta ha anunciado Llama 4, un modelo open-source con ventana de contexto de 10 millones de tokens. Supera a GPT-4o en varios benchmarks de comprensión larga. El modelo estará disponible en Hugging Face bajo licencia comercial."},
    {"titulo": "OpenAI integra búsqueda web nativa en ChatGPT",
     "fuente": "The Verge", "url": "https://theverge.com/chatgpt-search",
     "contenido": "ChatGPT ahora puede buscar en internet en tiempo real sin plugins adicionales. La función está disponible para usuarios Plus y Team. Los resultados incluyen citas de fuentes verificadas."},
    {"titulo": "Google DeepMind presenta AlphaFold 3 con predicción de interacciones moleculares",
     "fuente": "Nature", "url": "https://nature.com/alphafold3",
     "contenido": "La tercera versión de AlphaFold puede predecir interacciones entre proteínas, ADN, ARN y moléculas pequeñas. Esto abre nuevas posibilidades para el diseño de fármacos y la biología sintética."},
]

def modulo_resumir_articulo(articulo: dict) -> dict:
    """Módulo Make: HTTP module → Anthropic API para resumir."""
    # Esto replica exactamente lo que haría el módulo HTTP de Make
    payload = {
        "model": MODELO,
        "max_tokens": 250,
        "messages": [{
            "role": "user",
            "content": f"""Resume este artículo para una newsletter de IA en 3 puntos clave.
Usa un tono entusiasta pero profesional. En español.

Título: {articulo['titulo']}
Fuente: {articulo['fuente']}
Contenido: {articulo['contenido']}

Formato:
**{articulo['titulo']}**
• punto 1
• punto 2
• punto 3
[Leer más]({articulo['url']})"""
        }]
    }
    # En Make: el módulo HTTP POST envía este payload directamente
    resp = client.messages.create(**{k: v for k, v in payload.items() if k != "messages"},
                                   messages=payload["messages"])
    return {**articulo, "resumen_newsletter": resp.content[0].text}

def modulo_generar_newsletter(articulos_resumidos: list) -> str:
    """Módulo Make: ensambla el newsletter completo."""
    secciones = "\n\n".join(a["resumen_newsletter"] for a in articulos_resumidos)
    prompt = f"""Crea el encabezado de un newsletter semanal de IA con esta estructura:
- Saludo breve y fecha
- 1 frase resumen de la semana en IA
- (Las secciones ya están preparadas abajo)

Solo genera el encabezado, en español, máximo 3 líneas."""
    encabezado = client.messages.create(
        model=MODELO, max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    return f"{encabezado}\n\n---\n\n{secciones}"

print("=== PIPELINE RSS → NEWSLETTER ===")
resumidos = [modulo_resumir_articulo(art) for art in ARTICULOS_RSS]
newsletter = modulo_generar_newsletter(resumidos)
print(newsletter[:800] + "...")

with open("newsletter_generado.md", "w", encoding="utf-8") as f:
    f.write(newsletter)
print("\n✓ Newsletter guardado en newsletter_generado.md")

## 4. Zap: Google Sheets → Enriquecer → Actualizar

In [ ]:
# Simular filas de Google Sheets con clientes
CLIENTES_SHEETS = [
    {"id": "C001", "nombre": "Librería El Quijote", "sector": "", "descripcion": "Venta de libros físicos y eventos literarios"},
    {"id": "C002", "nombre": "FisioActiva", "sector": "", "descripcion": "Centro de fisioterapia deportiva y rehabilitación"},
    {"id": "C003", "nombre": "TechStartup SL", "sector": "", "descripcion": "Desarrollo de software a medida para pymes"},
]

def zap_enriquecer_cliente(cliente: dict) -> dict:
    """Simula el Zap: New Row in Sheets → HTTP Zapier → Update Row."""
    prompt = f"""Clasifica y enriquece este cliente para nuestro CRM.

Empresa: {cliente['nombre']}
Descripción: {cliente['descripcion']}

Responde SOLO con JSON:
{{"sector": "<sector principal>",
  "tamaño_estimado": "micropyme|pyme|mediana|grande",
  "digital_score": 1-10,
  "oportunidad_ia": "<principal caso de uso de IA para este negocio>",
  "tono_comunicacion": "formal|semiformal|cercano"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    enriquecimiento = json.loads(resp)
    return {**cliente, **enriquecimiento, "enriquecido_en": datetime.now().isoformat()}

print("=== ZAP: ENRIQUECIMIENTO DE CLIENTES ===")
clientes_enriquecidos = [zap_enriquecer_cliente(c) for c in CLIENTES_SHEETS]
df = pd.DataFrame(clientes_enriquecidos)
print(df[["nombre", "sector", "tamaño_estimado", "digital_score", "oportunidad_ia"]].to_string(index=False))
df.to_csv("clientes_enriquecidos.csv", index=False, encoding="utf-8")
print("\n✓ Datos enriquecidos exportados (simulando actualización en Google Sheets)")

## 5. Plantilla Make JSON para importar

In [ ]:
# Plantilla de escenario Make para importar directamente
# En Make: Importar escenario → pegar este JSON
plantilla_make = {
    "name": "Claude Email Classifier",
    "description": "Clasifica emails entrantes con Claude y los enruta según prioridad",
    "modules": [
        {"id": 1, "type": "gmail:watchEmails", "name": "Watch Gmail",
         "config": {"folder": "INBOX", "criteria": "is:unread"}},
        {"id": 2, "type": "http:makeRequest", "name": "Clasificar con Claude",
         "config": {
             "url": "https://api.anthropic.com/v1/messages",
             "method": "POST",
             "headers": [
                 {"name": "x-api-key", "value": "{{YOUR_ANTHROPIC_KEY}}"},
                 {"name": "anthropic-version", "value": "2023-06-01"},
                 {"name": "content-type", "value": "application/json"}
             ],
             "body": {
                 "model": "claude-haiku-4-5-20251001",
                 "max_tokens": 150,
                 "messages": [{"role": "user", "content":
                     "Clasifica este email. Asunto: {{1.subject}} Cuerpo: {{1.snippet}}. JSON: {categoria, prioridad, requiere_humano}"}]
             }
         }},
        {"id": 3, "type": "router", "name": "Router por prioridad",
         "routes": [
             {"condition": "{{2.content[0].text contains 'alta'}}", "next": 4},
             {"condition": "true", "next": 5}
         ]},
        {"id": 4, "type": "slack:createMessage", "name": "Notificar Slack (urgente)",
         "config": {"channel": "#soporte-urgente", "text": "🔴 Email urgente: {{1.subject}}"}},
        {"id": 5, "type": "gmail:createDraft", "name": "Crear borrador respuesta auto",
         "config": {"to": "{{1.from}}", "subject": "Re: {{1.subject}}"}},
    ]
}

with open("make_scenario_template.json", "w", encoding="utf-8") as f:
    json.dump(plantilla_make, f, ensure_ascii=False, indent=2)

print("✓ Plantilla Make guardada en make_scenario_template.json")
print("""
=== INSTRUCCIONES PARA MAKE ===

1. Ve a make.com → Crear nuevo escenario
2. Importar → pegar el contenido de make_scenario_template.json
3. Configurar conexiones:
   - Gmail: autorizar cuenta Google
   - HTTP module: añadir header x-api-key con tu API key de Anthropic
   - Slack: conectar workspace
4. Probar con 'Run once'
5. Activar para ejecución automática

=== INSTRUCCIONES PARA ZAPIER ===

1. Crea nuevo Zap con trigger: Gmail → New Email
2. Acción: Webhooks by Zapier → POST
   URL: https://api.anthropic.com/v1/messages
   Headers: x-api-key, anthropic-version, content-type
   Data: {model, messages, max_tokens}
3. Acción siguiente: parsear JSON de respuesta
4. Acción final: Gmail reply / Slack message
""")